In [10]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [11]:
max_features = 10000
max_len = 500
word_index=imdb.get_word_index()   # word_index is a dictionary that maps words to their corresponding integer indices in the IMDB dataset. This mapping is used to convert the text reviews into sequences of integers for input into the model. 
reverse_word_index={value:key for key,value in word_index.items()}  # reverse_word_index is a dictionary that maps integer indices back to their corresponding words. This is useful for decoding the sequences of integers back into human-readable text.

In [12]:
# rebuild the training architecture and load the saved weights
model = Sequential([
    Embedding(max_features, 128),
    SimpleRNN(128, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.build(input_shape=(None, max_len))
model.load_weights('simple_rnn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.get_weights()

[array([[ 0.1367298 , -0.21523704, -0.56519485, ...,  0.67025954,
         -0.6771662 ,  0.45869467],
        [ 0.05151305, -0.02758258, -0.01529273, ...,  0.03255276,
         -0.04284873,  0.00387234],
        [ 0.06765935, -0.05014161, -0.01983028, ...,  0.12649031,
          0.02801197, -0.01576779],
        ...,
        [ 0.03553689,  0.05185747,  0.02657442, ...,  0.02902407,
          0.02998407,  0.04511878],
        [ 0.0007906 , -0.00439954, -0.02963147, ..., -0.00579595,
          0.05300176, -0.00917181],
        [ 0.04824625,  0.01287592,  0.04406381, ..., -0.00440302,
          0.019628  ,  0.03919104]], dtype=float32),
 array([[-0.19066061, -0.11743016,  0.07294661, ..., -0.10833286,
          0.07233861,  0.05001818],
        [ 0.10294496, -0.12991405, -0.13077264, ..., -0.02636807,
          0.00373231, -0.10776638],
        [ 0.14259629, -0.17744152,  0.01196155, ...,  0.04656115,
          0.01982427,  0.04374127],
        ...,
        [-0.10265638, -0.02302573, -0.0

In [14]:
# step 2: Helper functions
# function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i-3,'?') for i in encoded_review])

# in above function - The function takes an encoded review (a list of integers) as input. It uses a list comprehension to iterate through each integer in the encoded review. For each integer, it looks up the corresponding word in the reverse_word_index dictionary, adjusting the index by subtracting 3 (since the first three indices are reserved for special tokens). If the integer is not found in the dictionary, it returns a '?' character. Finally, it joins the list of words into a single string and returns it.

# function to preprocess user input 
def preprocess_text(text):
    cleaned_text = re.sub(r"<br\s*/?>", " ", text.lower())
    cleaned_text = re.sub(r"[^a-z0-9']+", " ", cleaned_text)
    words = cleaned_text.split()
    encoded_review = [1] + [(word_index[word] + 3) if word in word_index else 2 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)
    return padded_review

# in above function - The preprocess_text function takes a raw text review as input. It first converts the text to lowercase and splits it into individual words. Then, it encodes each word into its corresponding integer index using the word_index dictionary, adding 3 to account for the reserved indices. If a word is not found in the dictionary, it defaults to an index of 2 (which typically represents an unknown token). Finally, it pads the encoded review to a maximum length of 500 using the sequence.pad_sequences function, ensuring that all reviews have the same length for input into the model.

In [15]:
## prediction function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction= model.predict(preprocessed_input, verbose=0)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment, prediction[0][0]

In [16]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment,score=predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Negative
Prediction Score: 0.42220205068588257
